# Establishing the Existence of an Assistant Basin

**Research question:** Does the transformer's own layer-by-layer dynamics actively restore the assistant representation when perturbed? If so, the assistant isn't just a geometric feature (an axis) — it's a **dynamical attractor** with self-correcting properties.

**Method:** Perturb the residual stream at various layers, in various directions and magnitudes. Measure whether subsequent layers pull the representation back toward the baseline (recovery) or let it diverge. Compare recovery rates for:
- **Assistant axis direction** (away from assistant)
- **Assistant axis direction** (toward assistant / overshoot)
- **Random directions** (control)

If the assistant direction recovers faster than random → specific restoring force exists.
If away and toward both recover symmetrically → true basin (not just directional bias).

**Built on:** [The Assistant Axis](https://arxiv.org/abs/2601.10387) (Lu et al., 2026)

## 1. Setup

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -q git+https://github.com/safety-research/assistant-axis.git
# !pip install -q torch transformers huggingface_hub matplotlib pandas scipy tqdm

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from basin_experiment import BasinExperiment, DEFAULT_PROMPTS
from basin_analysis import (
    plot_recovery_curves,
    plot_asymmetry,
    plot_basin_heatmap,
    plot_basin_width,
    test_directional_recovery,
    test_symmetry,
    print_summary,
    generate_all_plots,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_mem / 1e9
        print(f"  GPU {i}: {name} ({mem:.1f} GB)")

In [ ]:
# ---- Configuration ----
# Change MODEL_NAME to match your hardware:
#   - "google/gemma-2-27b-it"                → ~54GB, 1x A100-80GB or 2x A100-40GB
#   - "Qwen/Qwen3-32B"                       → ~64GB, 2x A100-40GB
#   - "meta-llama/Llama-3.3-70B-Instruct"    → ~140GB, 2x A100-80GB

MODEL_NAME = "google/gemma-2-27b-it"

# Experiment parameters
PERTURB_LAYERS = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45]  # Gemma 27B has 46 layers
ALPHAS = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
N_RANDOM_DIRS = 5
SEED = 42

In [ ]:
# Load model and axis
exp = BasinExperiment(MODEL_NAME)

print(f"Model: {MODEL_NAME}")
print(f"Layers: {exp.num_layers}")
print(f"Hidden dim: {exp.hidden_dim}")
print(f"Axis shape: {exp.axis.shape}")
print(f"Perturbation layers: {PERTURB_LAYERS}")
print(f"Alpha values: {ALPHAS}")
print(f"Directions per layer: 2 (assistant) + {N_RANDOM_DIRS} (random) = {2 + N_RANDOM_DIRS}")

## 2. Sanity Check

Before running the full experiment, verify on a single prompt:
1. Baseline activations are deterministic (same result on two runs)
2. Perturbation actually changes downstream activations
3. Recovery metrics look reasonable

In [ ]:
# Sanity check 1: Determinism
test_prompt = "What is the capital of France?"
input_ids = exp.tokenize(test_prompt)

baseline_1, logits_1 = exp.get_baseline_trajectory(input_ids)
baseline_2, logits_2 = exp.get_baseline_trajectory(input_ids)

max_diff = max(
    (baseline_1[l] - baseline_2[l]).abs().max().item()
    for l in baseline_1
)
print(f"Max activation difference between two baseline runs: {max_diff:.2e}")
assert max_diff < 1e-5, "Baseline is not deterministic!"
print("Baseline is deterministic.")

In [ ]:
# Sanity check 2: Perturbation has an effect
from basin_experiment import make_perturbation

test_layer = 22  # target layer for Gemma 27B
h_L = baseline_1[test_layer]
axis_dir = exp.axis[test_layer].float()
axis_dir = axis_dir / axis_dir.norm()

delta = make_perturbation(h_L, -axis_dir, alpha=0.5)  # away from assistant
perturbed_acts, perturbed_logits = exp.get_perturbed_trajectory(input_ids, test_layer, delta)

print(f"Perturbation norm: {delta.norm().item():.2f}")
print(f"Baseline activation norm at layer {test_layer}: {h_L.norm().item():.2f}")
print(f"Ratio: {delta.norm().item() / h_L.norm().item():.3f}")
print()

# Check downstream differences
for l in sorted(perturbed_acts.keys())[:5]:
    diff = (perturbed_acts[l] - baseline_1[l]).norm().item()
    base_norm = baseline_1[l].norm().item()
    print(f"  Layer {l}: distance = {diff:.4f}, normalized = {diff/base_norm:.4f}")

In [ ]:
# Sanity check 3: Quick recovery curve for one prompt, one layer
# Visualize whether distance grows or shrinks after perturbation

metrics = exp.compute_recovery_metrics(
    baseline_1, perturbed_acts, logits_1, perturbed_logits, test_layer
)

layers_after = [m["downstream_layer"] for m in metrics]
distances = [m["normalized_distance"] for m in metrics]
axis_gaps = [m["axis_projection_gap"] for m in metrics]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(layers_after, distances, "o-", color="#d62728", linewidth=2)
ax1.axvline(x=test_layer, color="gray", linestyle="--", alpha=0.5, label=f"Perturbation (layer {test_layer})")
ax1.set_xlabel("Layer")
ax1.set_ylabel("Normalized Distance from Baseline")
ax1.set_title("Does the perturbation shrink? (distance)")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(layers_after, axis_gaps, "o-", color="#2ca02c", linewidth=2)
ax2.axvline(x=test_layer, color="gray", linestyle="--", alpha=0.5)
ax2.axhline(y=0, color="black", linestyle="--", alpha=0.3)
ax2.set_xlabel("Layer")
ax2.set_ylabel("Axis Projection Gap")
ax2.set_title("Does the gap close? (axis projection)")
ax2.grid(True, alpha=0.3)

fig.suptitle(f"Sanity Check: Single prompt, perturbed away from assistant at layer {test_layer}, alpha=0.5", fontsize=12)
fig.tight_layout()
plt.show()

print(f"\nKL divergence (output): {metrics[0]['kl_divergence']:.4f}")
print(f"Top-1 token preserved: {bool(metrics[0]['top1_preserved'])}")

## 3. Full Experiment

Run the complete perturbation-recovery protocol across all prompts, layers, directions, and magnitudes. This is the main compute-intensive step (~4-5 hours on A100-80GB with Gemma 27B).

**What's being computed:**
- 50 prompts × 10 perturbation layers × 7 directions × 7 magnitudes = ~24,500 perturbed forward passes
- Plus 50 baseline forward passes
- At each, we measure recovery at all downstream layers

In [ ]:
# Use first 50 prompts (increase to 100 for full run)
prompts = DEFAULT_PROMPTS[:50]

print(f"Running experiment:")
print(f"  Prompts: {len(prompts)}")
print(f"  Perturbation layers: {PERTURB_LAYERS}")
print(f"  Alphas: {ALPHAS}")
print(f"  Directions: 2 (assistant) + {N_RANDOM_DIRS} (random)")
n_passes = len(prompts) * len(PERTURB_LAYERS) * (2 + N_RANDOM_DIRS) * len(ALPHAS) + len(prompts)
print(f"  Total forward passes: ~{n_passes:,}")

df = exp.run_experiment(
    prompts=prompts,
    perturb_layers=PERTURB_LAYERS,
    alphas=ALPHAS,
    n_random_dirs=N_RANDOM_DIRS,
    seed=SEED,
)

print(f"\nResults shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Save results
df.to_parquet(RESULTS_DIR / "basin_results.parquet", index=False)
print(f"Results saved to {RESULTS_DIR / 'basin_results.parquet'}")

# To reload later:
# df = pd.read_parquet(RESULTS_DIR / "basin_results.parquet")

## 4. Analysis & Visualization

### 4.1 Recovery Curves (the money plot)

The key question: does the assistant axis perturbation recover faster than random?
- **Red** = pushed away from assistant
- **Green** = pushed toward assistant (overshoot)
- **Gray** = random direction (control)

If red/green lines fall below gray → the assistant direction has a specific restoring force.

In [ ]:
# Recovery curves at different magnitudes (averaged over all perturbation layers)
for alpha in [0.1, 0.5, 1.0]:
    fig, ax = plot_recovery_curves(df, alpha=alpha)
    plt.show()

In [ ]:
# Recovery curves at specific perturbation layers (alpha=0.5)
# Compare early, mid, and late layers
key_layers = [
    PERTURB_LAYERS[0],                        # early
    PERTURB_LAYERS[len(PERTURB_LAYERS) // 2],  # mid
    PERTURB_LAYERS[-2],                        # late (not last — need downstream room)
]

for layer in key_layers:
    fig, ax = plot_recovery_curves(df, alpha=0.5, perturb_layer=layer)
    plt.show()

### 4.2 Asymmetry Test

Is the recovery symmetric? If both "away from assistant" and "toward assistant" converge to the same point (gap → 0), that's a **true basin**. If only one direction recovers, it's a directional bias.

In [ ]:
# Asymmetry plots
for alpha in [0.1, 0.5, 1.0]:
    fig, ax = plot_asymmetry(df, alpha=alpha)
    plt.show()

### 4.3 Basin Heatmap

Where is the basin deepest? Green = recovery (inside basin), Red = divergence (outside).

Compare the assistant axis heatmap to the random direction heatmap. If the assistant map is greener, there's a direction-specific basin.

In [ ]:
# Basin heatmaps
fig1, ax1 = plot_basin_heatmap(df, "assistant_away")
plt.show()

fig2, ax2 = plot_basin_heatmap(df, "random")
plt.show()

### 4.4 Basin Width Across Layers

The critical alpha (largest perturbation that still recovers) at each layer.
If assistant_away has a wider basin radius than random at mid-layers, the network specifically tolerates perturbation along the assistant direction.

In [ ]:
# Basin width comparison
fig, ax = plot_basin_width(df)
plt.show()

## 5. Statistical Tests

Two key tests:
1. **Directional recovery test:** Does the assistant axis direction recover significantly faster than random? (paired t-test)
2. **Symmetry test:** Do away-from and toward-assistant perturbations converge symmetrically? (paired t-test on final gap magnitudes)

In [ ]:
# Run statistical tests at multiple alpha values
for alpha in [0.1, 0.5, 1.0]:
    print(f"\n{'='*50}")
    print(f"Alpha = {alpha}")
    print(f"{'='*50}")
    
    directional = test_directional_recovery(df, alpha)
    print(f"\nDirectional Recovery Test:")
    print(f"  Away recovery score:   {directional['mean_away']:.4f}")
    print(f"  Random recovery score: {directional['mean_random']:.4f}")
    print(f"  t={directional['t_statistic']:.3f}, p={directional['p_value']:.2e}, d={directional['cohens_d']:.3f}")
    print(f"  Significant: {directional['significant_at_0.05']}")
    
    symmetry = test_symmetry(df, alpha)
    print(f"\nSymmetry Test:")
    print(f"  Final gap (away):   {symmetry['mean_gap_away']:.4f}")
    print(f"  Final gap (toward): {symmetry['mean_gap_toward']:.4f}")
    print(f"  t={symmetry['t_statistic']:.3f}, p={symmetry['p_value']:.2e}")
    print(f"  Symmetric: {symmetry['symmetric_at_0.05']}")

## 6. Summary & Interpretation

In [ ]:
# Print full summary with conclusion
print_summary(df, alpha=0.5)

In [ ]:
# Save all plots to results/
generate_all_plots(df, output_dir="results")
print("Done. Check results/ for all saved plots.")

## 7. Interpretation Guide

### Reading the results:

| What you see | What it means |
|---|---|
| Red line below gray in recovery curves | Assistant direction recovers faster than random → **restoring force exists** |
| Red and green converge to same level | Symmetric recovery → **true basin** (not just directional bias) |
| Green basin heatmap at mid-layers, red at edges | Basin is deepest in mid-layers (consistent with Fernando & Guitchounts 2025) |
| Assistant basin radius > random basin radius | The network specifically tolerates perturbation along the assistant direction |

### What the conclusions mean:

- **"Evidence supports the existence of an assistant basin"** → The transformer actively self-corrects toward assistant-like representations. This is a stronger claim than the axis paper, which only showed a geometric feature. Safety implication: the model has built-in resistance to persona drift.

- **"Directional restoring force exists, but asymmetric"** → There's a restoring force, but it only works in one direction (e.g., recovering from being pushed away, but not from overshoot). This is a directional bias, not a true attractor. Safety implication: the model resists being pushed away from assistant mode, but overshooting (being "too helpful") is stable.

- **"No evidence for an assistant-specific basin"** → Recovery (if any) is a generic property of the architecture (LayerNorm, etc.), not specific to assistant-ness. The assistant axis is just a geometric feature with no dynamical significance. Safety implication: the axis can be used for measurement but provides no inherent stability.

### Next steps:
1. If positive: Repeat on Qwen 3 32B and Llama 3.3 70B (pre-computed axes available)
2. Map basin shape in more directions (refusal vector, truth direction, PCA components)
3. Compare base vs instruct models: does RLHF deepen the basin?
4. Connect to tuned lens: does the prediction trajectory recover incrementally or chaotically?